# VESTIGIUM — Reproducible Hotspot EDA

NASA Space Apps Challenge 2026 · Team Ga Tidur

This is the **runnable, repository-native summary EDA**. It uses the verified ingestion outputs in `data/processed/data-engineering`. The original 43-cell full report with 17 figures has been preserved as `VESTIGIUM_EDA_FULL_REFERENCE.ipynb`. That extended notebook requires its own historical EDA artifact bundle.

Unit: one satellite-detected thermal pixel; *not* one unique fire or ignition cause.

## 1. Reproducible setup

Run `src/pipelines/build_pipeline.py` first using NASA FIRMS source ZIP, as shown in `scripts/README.md`.

In [ ]:
from pathlib import Path
import json, sqlite3
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p/'README.md').is_file() and (p/'data').is_dir()), None)
assert ROOT is not None, 'Open notebook inside the VESTIGIUM repository'
OUT = ROOT/'data'/'processed'/'data-engineering'
qa = json.loads((OUT/'quality_report.json').read_text(encoding='utf8'))
db=OUT/'vestigium_hotspots.sqlite'
assert db.exists(), 'Run the ingestion pipeline before EDA'
conn=sqlite3.connect(f'file:{db.resolve()}?mode=ro',uri=True)
display(pd.DataFrame([qa]))

## 2. Source inventory and exclusions

The Angola file is excluded by filename. NASA FIRMS source columns and sensor capabilities vary over time.

In [ ]:
inventory=pd.read_sql_query('SELECT filename,sensor,nominal_year,source_rows,accepted_type0,excluded_non_vegetation,rejected_invalid,first_date_utc,last_date_utc FROM source_files ORDER BY sensor,nominal_year',conn)
display(inventory.groupby('sensor')[['source_rows','accepted_type0','excluded_non_vegetation','rejected_invalid']].sum())
print('Indonesia CSV files:',len(inventory))
print('Source detections:',inventory.source_rows.sum())
print('Accepted type=0:',inventory.accepted_type0.sum())
assert int(inventory.accepted_type0.sum())==qa['accepted_presumed_vegetation_detections']

## 3. Temporal aggregates are sensor-specific

In [ ]:
daily=pd.read_csv(OUT/'daily_sensor.csv')
monthly=pd.read_csv(OUT/'monthly_sensor.csv')
annual=pd.read_csv(OUT/'annual_sensor.csv')
expected=qa['accepted_presumed_vegetation_detections']
for name,d in [('daily',daily),('monthly',monthly),('annual',annual)]:
    observed=int(d.detection_count.sum())
    print(name,observed)
    assert observed==expected
display(annual.pivot(index='year',columns='sensor',values='detection_count').tail())

## 4. Annual trends

Raw MODIS and VIIRS observations are **not a homogeneous time series**.

In [ ]:
fig,ax=plt.subplots(figsize=(11,4))
for sensor,g in annual.groupby('sensor'):
    ax.plot(g.year,g.detection_count,marker='.',label=sensor)
ax.set(title='Annual satellite hotspot detections by sensor',
       xlabel='UTC year',ylabel='Detected thermal pixels')
ax.legend();plt.show()

## 5. Monthly seasonality and year-specific peaks

In [ ]:
display(monthly.nlargest(15,'detection_count')[['year','month','sensor','detection_count','frp_sum_mw']])
for year in (2015,2019):
    p=monthly[monthly.year.eq(year)].pivot(index='month',columns='sensor',values='detection_count')
    p.plot(figsize=(10,4),title=f'{year} monthly hotspot detections by satellite')
    plt.ylabel('Detected thermal pixels');plt.xlabel('UTC month');plt.show()

## 6. Detection-level FRP by sensor

FRP is pixel-level radiative power measured in MW; it does not directly measure burned area.

In [ ]:
frp=pd.read_sql_query('SELECT sensor,COUNT(*) AS pixel_count, AVG(frp_mw) AS mean_frp_mw, MIN(frp_mw) AS min_frp_mw, MAX(frp_mw) AS max_frp_mw FROM hotspots GROUP BY sensor',conn)
display(frp)

## 7. Geographic activity (0.25-degree raster approximation)

Grid cells are degree-based and therefore have different physical area across latitudes.

In [ ]:
grid=pd.read_csv(OUT/'annual_grid.csv')
assert int(grid.detection_count.sum()) == expected
a=grid[(grid.year.eq(2015))&(grid.sensor.eq('MODIS'))]
fig,ax=plt.subplots(figsize=(11,5))
ax.scatter(a.grid_center_lon,a.grid_center_lat,
           s=a.detection_count.clip(1,500)/8,alpha=.25,c='firebrick')
ax.set(title='MODIS 2015 grid-cell detection counts (not individual fires)',
       xlabel='Longitude',ylabel='Latitude')
plt.show()

## 8. Quality, interpretation, and next steps

In [ ]:
observed=conn.execute('SELECT COUNT(*) FROM hotspots').fetchone()[0]
assert observed==expected
print('Source conservation: PASS',observed)
print('Excluded other FIRMS type categories:',qa['excluded_non_vegetation_detections'])
print('Rejected invalid presumed-vegetation rows:',qa['rejected_invalid_rows'])
print('Sensor-specific confidence is stored without converting VIIRS classes to MODIS percentages')
print('Scientific limitations: sensor differences, unmeasured overpass effort, clouds and reporting versions')
conn.close()